In [1]:
# extract from wannier_info.
# dic = {k: wannier_info[k] for k in ["A", "atoms_frac", "nw2l", "nw2m", "nw2r", "nw2s"]}
Te_dict = {
    'A': [[4.458, 0.0, 0.0], [-2.2290001, 3.8607414, 0.0], [0.0, 0.0, 5.925]],
    'atoms_frac': {
        ('Te', 1): [0.274, 0.0, 0.3333333333333],
        ('Te', 2): [0.726, 0.726, 0.0],
        ('Te', 3): [0.0, 0.274, 0.6666666666666],
    },
    'nw2l': [1, 1, 1, 1, 1, 1, 1, 1, 1],
    'nw2m': [1, 2, 3, 1, 2, 3, 1, 2, 3],
    'nw2r': [1, 1, 1, 1, 1, 1, 1, 1, 1],
    'nw2s': [0, 0, 0, 0, 0, 0, 0, 0, 0],
    #
    "site_dict":{
        ('A', 1): [0.274, 0.0, 0.3333333333333333],
        ('A', 2): [0.0, 0.274, 0.6666666666666666],
        ('A', 3): [0.726, 0.726, 0.0]
    },
    "no": 152,
}
graphene_dict = {
    'A': [[2.435000000005462, 0.0, 0.0], [-1.217500000002731, 2.1087718582198383, 0.0], [0.0, 0.0, 9.740000000021848]],
    'atoms_frac': {
        ('C', 1): [0.6666666666666666, 0.3333333333333333, 0.0],
        ('C', 2): [0.3333333333333333, 0.6666666666666666, 0.0]
    },
    'nw2l': [1, 1],
    'nw2m': [1, 1],
    'nw2r': [1, 1],
    'nw2s': [0, 0],
    #
    "site_dict":{
        ('A', 1): [0.3333333333333333, 0.6666666666666666, 0.0],
        ('A', 2): [0.6666666666666666, 0.3333333333333333, 0.0]
    },
    "no": 191,
}

import numpy as np
import spglib
from multipie import Group
from multipie.util.util_wannier import _convert_w90_orbital

SYMPREC = 1e-4
def find_sg(A, atoms_frac, symprec=SYMPREC):
    positions = list(atoms_frac.values())
    elements = list(set([i[0] for i in atoms_frac.keys()]))
    elements = {i: no for no, i in enumerate(elements)}
    numbers = [elements[i[0]] for i in atoms_frac.keys()]
    cell = (A, positions, numbers)

    dataset = spglib.get_symmetry_dataset(cell, symprec=symprec)
    if dataset:
        return dataset.number
    else:
        return None

def find_vector_index(vector_list, vector, symprec=SYMPREC):
    for idx, v in enumerate(vector_list):
        if np.allclose(v, vector, rtol=0, atol=symprec):
            return idx
    return None

def get_or_add_vector(existing_list, target_vector, symprec=SYMPREC):
    target_arr = np.asarray(target_vector).reshape(-1)

    for idx, v in enumerate(existing_list):
        if np.allclose(v, target_arr, rtol=0, atol=symprec):
            return idx

    existing_list.append(target_arr)
    return len(existing_list) - 1

def create_ket_wannier(wannier_info):
    # create orbital list.
    orbital_info = [wannier_info[key] for key in ("nw2l", "nw2m", "nw2r", "nw2s")]
    orbital_list = []
    for l, m, r, s in zip(*orbital_info):
        comp, orbital = _convert_w90_orbital(l, m, r, s)
        orbital_list.append((l, comp, orbital))

    # determine space group by spglib.
    space_group_no = find_sg(wannier_info["A"], wannier_info["atoms_frac"])
    group = Group(space_group_no)

    # create atom site-cluster info.
    existing_list = []
    site_cluster = {}
    atom_info = []
    for (atom, _), pos in d["atoms_frac"].items():
        wp, sites = group.find_wyckoff_site(pos)
        idx = get_or_add_vector(existing_list, sites)
        if (atom,wp,idx) not in site_cluster.keys():
            site_cluster[(atom,wp,idx)] = sites
        atom_info.append((atom, wp, idx, pos))

    # create ket info. (atom, sublattice, rank, component, orbital), frac_position, (wycokff, multiplicity).
    ket = []
    for (atom, wp, idx, pos), (l, comp, orbital) in zip(atom_info,orbital_list):
        sites = site_cluster[(atom,wp,idx)]
        site_idx = find_vector_index(sites, pos)
        ket.append( ((atom, site_idx+1, l, comp, orbital), pos, (wp, idx+1)) )

    return space_group_no, str(group), site_cluster, ket



In [2]:
for d in [Te_dict, graphene_dict]:
    sg_no, sg_name, site_cluster, ket = create_ket_wannier(d)
    print("space group =", sg_no, sg_name)
    print("=== site cluster ===")
    for k, s in site_cluster.items():
        print(k, s.tolist())
    print("=== ket info. ===")
    for ket_info, pos, wp_idx in ket:
        print(ket_info, pos, wp_idx)
    print()

space group = 152 D3^4
=== site cluster ===
('Te', '3a', 0) [[0.274, 0.0, 0.3333333333333333], [0.0, 0.274, 0.6666666666666666], [0.726, 0.726, 0.0]]
=== ket info. ===
('Te', 1, 1, 2, 'pz') [0.274, 0.0, 0.3333333333333] ('3a', 1)
('Te', 3, 1, 0, 'px') [0.726, 0.726, 0.0] ('3a', 1)
('Te', 2, 1, 1, 'py') [0.0, 0.274, 0.6666666666666] ('3a', 1)

space group = 191 D6h^1
=== site cluster ===
('C', '2c', 0) [[0.3333333333333333, 0.6666666666666666, 0.0], [0.6666666666666666, 0.3333333333333333, 0.0]]
=== ket info. ===
('C', 2, 1, 2, 'pz') [0.6666666666666666, 0.3333333333333333, 0.0] ('2c', 1)
('C', 1, 1, 2, 'pz') [0.3333333333333333, 0.6666666666666666, 0.0] ('2c', 1)

